# Cylinder flow MeshGraphNet training

`case_000.pt` と `case_001.pt` で学習し、`case_002.pt` で検証します。

上から順に実行すると、データの読み込み、正規化、モデル作成、学習まで進みます。

## 1. ライブラリと設定

設定値は、実際に使う行へ直接書いてあります。

In [1]:
from pathlib import Path
import json
import os

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

os.environ["PHYSICSNEMO_FORCE_TE"] = "False"
from physicsnemo.models.meshgraphnet import MeshGraphNet

DATA_DIR = Path("data/preprocessed")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

INTERIOR = 0
NUM_NODE_TYPES = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print("device:", DEVICE)

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: cuda


## 2. 訓練用と検証用のデータを読み込む

ケースは固定で、`case_000.pt` と `case_001.pt` を訓練用、`case_002.pt` を検証用にします。

In [2]:
train_cases = [
    torch.load(DATA_DIR / "case_000.pt", map_location="cpu", weights_only=True),
    torch.load(DATA_DIR / "case_001.pt", map_location="cpu", weights_only=True),
]
validation_cases = [
    torch.load(DATA_DIR / "case_002.pt", map_location="cpu", weights_only=True),
]

print("訓練ケース:", [case["case_id"] for case in train_cases])
print("検証ケース:", [case["case_id"] for case in validation_cases])
print("state shape:", tuple(train_cases[0]["state"].shape))

訓練ケース: ['000', '001']
検証ケース: ['002']
state shape: (501, 5109, 3)


## 3. 訓練データの統計量で正規化する

検証データを使わず、訓練ケースだけから平均と標準偏差を計算します。

In [3]:
def edge_features(case):
    source, destination = case["edge_index"]
    displacement = case["pos"][source] - case["pos"][destination]
    length = torch.linalg.vector_norm(displacement, dim=-1, keepdim=True)
    return torch.cat((displacement, length), dim=-1)


def mean_std(values):
    values = torch.cat([value.reshape(-1, value.shape[-1]) for value in values])
    return values.mean(0), values.std(0).clamp_min(1.0e-8)

velocity_mean, velocity_std = mean_std([case["state"][:, :, :2] for case in train_cases])
velocity_delta_mean, velocity_delta_std = mean_std([
    case["state"][1:, :, :2] - case["state"][:-1, :, :2] for case in train_cases
])
pressure_mean, pressure_std = mean_std([case["state"][1:, :, 2:] for case in train_cases])
edge_mean, edge_std = mean_std([edge_features(case) for case in train_cases])

normalizer = {
    "velocity_mean": velocity_mean,
    "velocity_std": velocity_std,
    "velocity_delta_mean": velocity_delta_mean,
    "velocity_delta_std": velocity_delta_std,
    "pressure_mean": pressure_mean,
    "pressure_std": pressure_std,
    "edge_mean": edge_mean,
    "edge_std": edge_std,
}
with (OUTPUT_DIR / "normalization.json").open("w", encoding="utf-8") as file:
    json.dump({name: value.tolist() for name, value in normalizer.items()}, file, indent=2)

## 4. Dataset、DataLoader、MeshGraphNetを作る

1つの時刻から次の時刻を予測するデータセットを作ります。

In [ ]:
class CylinderDataset(Dataset):
    def __init__(self, cases):
        self.samples = [(case, time) for case in cases for time in range(case["state"].shape[0] - 1)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        case, time = self.samples[index]
        state = case["state"]
        velocity = (state[time, :, :2] - normalizer["velocity_mean"]) / normalizer["velocity_std"]
        delta = (state[time + 1, :, :2] - state[time, :, :2] - normalizer["velocity_delta_mean"]) / normalizer["velocity_delta_std"]
        pressure = (state[time + 1, :, 2:] - normalizer["pressure_mean"]) / normalizer["pressure_std"]
        node_type = F.one_hot(case["node_type"], NUM_NODE_TYPES).float()
        edge = (edge_features(case) - normalizer["edge_mean"]) / normalizer["edge_std"]
        return Data(
            x=torch.cat((velocity, node_type), dim=-1),
            edge_index=case["edge_index"],
            edge_attr=edge,
            y=torch.cat((delta, pressure), dim=-1),
            pos=case["pos"],
        )


train_loader = DataLoader(CylinderDataset(train_cases), batch_size=1, shuffle=True)
validation_loader = DataLoader(CylinderDataset(validation_cases), batch_size=1)

sample = next(iter(train_loader))
print(sample.x.shape)  # 
print(sample.y.shape)

model = MeshGraphNet(
    input_dim_nodes=6,
    input_dim_edges=3,
    output_dim=3,
    processor_size=15,
    hidden_dim_processor=128,
    hidden_dim_node_encoder=128,
    hidden_dim_edge_encoder=128,
    hidden_dim_node_decoder=128,
    aggregation="sum",
    mlp_activation_fn="relu",
    norm_type="LayerNorm",
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)


torch.Size([5109, 6])
torch.Size([5109, 3])


## 5. 損失関数と1エポックの処理

モデルの出力は `(速度差 u, 速度差 v, 圧力)` です。

In [ ]:
def loss_values(prediction, target):
    return {
        "loss": F.mse_loss(prediction, target),
        "velocity": F.mse_loss(prediction[:, :2], target[:, :2]),
        "pressure": F.mse_loss(prediction[:, 2:], target[:, 2:]),
    }


def run_epoch(loader, training):
    model.train(training)
    total = {"loss": 0.0, "velocity": 0.0, "pressure": 0.0}
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for graph in loader:
            print(graph)
            quit()
            graph = graph.to(DEVICE)
            prediction = model(graph.x, graph.edge_attr, graph)
            losses = loss_values(prediction, graph.y)
            if training:
                optimizer.zero_grad()
                losses["loss"].backward()
                optimizer.step()
            for name, value in losses.items():
                total[name] += value.item()
    return {name: value / len(loader) for name, value in total.items()}

## 6. 学習を実行する

各エポックで訓練と検証を行い、検証損失が最小のモデルを保存します。

In [6]:
history = []
best_validation_loss = float("inf")

for epoch in range(1, 11):
    train_metrics = run_epoch(train_loader, training=True)
    validation_metrics = run_epoch(validation_loader, training=False)
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "validation_loss": validation_metrics["loss"],
        "train_velocity": train_metrics["velocity"],
        "validation_velocity": validation_metrics["velocity"],
        "train_pressure": train_metrics["pressure"],
        "validation_pressure": validation_metrics["pressure"],
    }
    history.append(row)
    print(row)

    if validation_metrics["loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["loss"]
        torch.save(
            {
                "model": model.state_dict(),
                "normalizer": {name: value.tolist() for name, value in normalizer.items()},
                "epoch": epoch,
            },
            OUTPUT_DIR / "best.pt",
        )

with (OUTPUT_DIR / "history.json").open("w", encoding="utf-8") as file:
    json.dump(history, file, indent=2)

print("保存完了:", OUTPUT_DIR / "best.pt")

{'epoch': 1, 'train_loss': 0.296545810488984, 'validation_loss': 0.7678567215800285, 'train_velocity': 0.32974788130261, 'validation_velocity': 0.7390671104490757, 'train_pressure': 0.2301416485402733, 'validation_pressure': 0.8254359411001205}
{'epoch': 2, 'train_loss': 0.21357957746088505, 'validation_loss': 0.7336165938973427, 'train_velocity': 0.23605433874391019, 'validation_velocity': 0.6978670406341553, 'train_pressure': 0.1686300328411162, 'validation_pressure': 0.8051157044172287}
{'epoch': 3, 'train_loss': 0.14089694222807883, 'validation_loss': 0.6785863220691681, 'train_velocity': 0.15861257013399155, 'validation_velocity': 0.6244019293785095, 'train_pressure': 0.1054656791575253, 'validation_pressure': 0.7869550969600677}
{'epoch': 4, 'train_loss': 0.12021370091103017, 'validation_loss': 0.7242843571305275, 'train_velocity': 0.13781154632847756, 'validation_velocity': 0.6855066295862198, 'train_pressure': 0.08501799312699586, 'validation_pressure': 0.8018398156166077}
{'ep